# TROPOMI SRON SIF L2>L3

already a column averaged product - gridded data 

Using the HARP package to convert this to a level 3 TROPOMI product on a regular grid
- 0.25 x 0.25 degree grid

- https://www.youtube.com/watch?v=CE6BeLPORIE
- https://stcorp.github.io/harp/doc/html/index.html 

In [ ]:
import datetime
import sys
import contextlib
from pathlib import Path
import pandas as pd
import xarray as xr
import numpy as np
import glob
import matplotlib.pyplot as plt
import harp
import itertools



In [ ]:
#H2O column conversion

#molec/cm**2

#cm**2 --> kg
#(pressure in Pa)*( area in m**2)/(gravity in m/s**2)
P = 101300 #should this be 101325? 
A = 10**(-4)
g = 9.81 

conv_denom = P*A/g

#molec --> g
#(number of molecules)*(atomic weight in g/mol)/(avogadro number in molec/mol)
AW = 18 #should be 18.015?
AV = 6.02214076*10**23

In [ ]:
export_path = '/Volumes/New_5TB/ESA_F4R/SIF_L3/'
for Y in range(2018,2025):
    pathlist = glob.glob("/Volumes/New_5TB/ESA_F4R/SIF_L2/*.nc")
    print(pathlist)
    for t,file in enumerate(pathlist):
        harp_L2_L3 = harp.import_product(file, operations=" \
            derive(datetime_stop{time}); \
            latitude > -15.5 [degree_north] ; latitude < 12.5 [degree_north] ; longitude > 7.5 [degree_east] ; longitude < 31.5 [degree_east]; \
            bin_spatial(112,-15,0.25,96,8,0.25); \
            derive(latitude{latitude}); derive(longitude{longitude})")
        
        print(harp_L2_L3)
        export_folder = "{export_path}/{name}".format(export_path=export_path, name=file.split('/')[-1].replace('L2','L3'))
        harp.export_product(harp_L2_L3,export_folder,file_format='netcdf')